## 2: AgentCore Identity

**What you'll learn:** Configure secure authentication using Amazon Cognito and JWT tokens for agent access control.

**Why it matters:** Production agents need authentication to prevent unauthorized access and enable user-specific operations. IAM roles define what your agent can access.

**Real-world value:** Identity ensures only authorized users invoke agents and enables personalized responses based on user identity—critical for enterprise deployments.

**Analogy:** Like a building security system: Cognito is the front desk issuing visitor badges (JWT tokens), and IAM roles are the access cards determining which floors you can visit.

![Identity](images/Identity.png)

---

**Prerequisites:** Completed Notebook 1, AWS permissions for Cognito

### Import Required Libraries

In [1]:
import os


# Set the AWS profile to match your shell environment
os.environ['AWS_PROFILE'] = 'workshop-profile'

# Now import boto3 and it will use the correct profile
import json
import boto3
from strands import Agent
from strands.models import BedrockModel

print(f"Import Required Libraries")
session = boto3.Session()
sts = session.client('sts')
identity = sts.get_caller_identity()
account_id = identity['Account']
region = session.region_name or 'us-west-2'

print(f"✅ Account ID: {account_id}")
print(f"✅ Region: {region}")


Import Required Libraries
✅ Account ID: 625579972148
✅ Region: us-west-2


In [2]:
import boto3
from utils.identity_ssm_utils import setup_cognito_user_pool, reauthenticate_user

# Get AWS session information
session = boto3.Session()
region = session.region_name or 'us-west-2'

sts = session.client('sts')
identity = sts.get_caller_identity()
account_id = identity['Account']

print(f"Account ID: {account_id}")
print(f"Region: {region}")

Account ID: 625579972148
Region: us-west-2


### Setting Up Amazon Cognito User Pool

Amazon Cognito provides user authentication for AgentCore Runtime and AgentCore Gateway. This creates a User Pool, App Client, and test user.

**Note:** Access tokens are valid for 2 hours. Use `reauthenticate_user()` to generate new tokens.

In [3]:
# Diagnostic cell - run this first to see the actual error
import boto3

try:
    # Test Cognito permissions
    cognito_client = boto3.client('cognito-idp', region_name=region)
    
    # Try to list user pools (this will fail if no permissions)
    response = cognito_client.list_user_pools(MaxResults=1)
    print("✅ Cognito permissions OK")
    
    # Test Secrets Manager permissions
    secrets_client = boto3.client('secretsmanager', region_name=region)
    try:
        secrets_client.describe_secret(SecretId='returns_refunds_agent')
        print("ℹ️ Secret 'returns_refunds_agent' already exists")
    except secrets_client.exceptions.ResourceNotFoundException:
        print("✅ Secrets Manager permissions OK (secret doesn't exist yet)")
    
    print("\nNow try running the setup_cognito_user_pool() function...")
    
except Exception as e:
    print(f"❌ Permission check failed: {str(e)}")
    print("\nYou may need additional IAM permissions.")


✅ Cognito permissions OK
✅ Secrets Manager permissions OK (secret doesn't exist yet)

Now try running the setup_cognito_user_pool() function...


In [4]:
print("Setting up Amazon Cognito user pool...")
cognito_config = setup_cognito_user_pool()
print("Cognito setup completed ✓")

# Display important configuration details
print("\n" + "=" * 80)
print("Cognito Configuration:")
print("=" * 80)
print(f"Pool ID: {cognito_config.get('pool_id')}")
print(f"Client ID: {cognito_config.get('client_id')}")
print(f"Discovery URL: {cognito_config.get('discovery_url')}")
print(f"\nBearer Token (valid for 2 hours):")
print(f"{cognito_config.get('bearer_token')[:50]}...")
print("=" * 80)

Setting up Amazon Cognito user pool...
Creating resource server for OAuth...
✅ Resource server created
Creating user pool domain: returns-refunds-agent-1769152548
✅ User pool domain created
{'UserPoolId': 'us-west-2_QuW2OOBnZ', 'ClientName': 'ReturnsRefundsAgentPoolClient', 'ClientId': '6dacsfqdd4b9imggc4dnlsiebv', 'ClientSecret': '1p3ief1umuhq6d73mp8569uvp5o2qcnb7rf0mdf8oorv54vco4uk', 'LastModifiedDate': datetime.datetime(2026, 1, 23, 2, 15, 53, 958000, tzinfo=tzlocal()), 'CreationDate': datetime.datetime(2026, 1, 23, 2, 15, 53, 958000, tzinfo=tzlocal()), 'RefreshTokenValidity': 30, 'TokenValidityUnits': {}, 'ExplicitAuthFlows': ['ALLOW_USER_PASSWORD_AUTH', 'ALLOW_USER_SRP_AUTH', 'ALLOW_REFRESH_TOKEN_AUTH'], 'SupportedIdentityProviders': ['COGNITO'], 'AllowedOAuthFlows': ['client_credentials'], 'AllowedOAuthScopes': ['workshop-api/write', 'workshop-api/read'], 'AllowedOAuthFlowsUserPoolClient': True, 'EnableTokenRevocation': True, 'EnablePropagateAdditionalUserContextData': False, 'Au

### Creating IAM Execution Role

AgentCore Runtime needs an IAM role with permissions to access Bedrock models, Knowledge Bases, Memory, and CloudWatch logs.

In [5]:
from utils.identity_ssm_utils import create_agentcore_runtime_execution_role

# Create execution role for AgentCore Runtime
agentcore_runtime_execution_role = create_agentcore_runtime_execution_role()

print(f"\nExecution Role ARN: {agentcore_runtime_execution_role}")

✅ Created IAM role: ReturnsRefundsAssistantBedrockAgentCoreRole-us-west-2
Role ARN: arn:aws:iam::625579972148:role/ReturnsRefundsAssistantBedrockAgentCoreRole-us-west-2
✅ Created policy: ReturnsRefundsAssistantBedrockAgentCorePolicy-us-west-2
✅ Attached policy to role
Policy ARN: arn:aws:iam::625579972148:policy/ReturnsRefundsAssistantBedrockAgentCorePolicy-us-west-2

Execution Role ARN: arn:aws:iam::625579972148:role/ReturnsRefundsAssistantBedrockAgentCoreRole-us-west-2


### Refreshing Authentication Tokens

Cognito tokens expire after 2 hours. Use this cell to generate a fresh token:

In [ ]:
# Refresh the bearer token
bearer_token = reauthenticate_user(
    cognito_config.get("client_id"),
    cognito_config.get("client_secret")
)

print("✅ New bearer token generated!")
print(f"Token (first 50 chars): {bearer_token[:50]}...")
print("\n⏰ This token is valid for 2 hours from now.")

### Saving Configuration for Later Use

Store the Cognito configuration for use in other notebooks:

In [6]:
import json

# Save configuration to a file for easy access
config_data = {
    "pool_id": cognito_config.get("pool_id"),
    "client_id": cognito_config.get("client_id"),
    "client_secret": cognito_config.get("client_secret"),
    "discovery_url": cognito_config.get("discovery_url"),
    "execution_role": agentcore_runtime_execution_role
}

with open('cognito_config.json', 'w') as f:
    json.dump(config_data, f, indent=2)

print("✅ Configuration saved to cognito_config.json")

✅ Configuration saved to cognito_config.json


### Summary

You've configured Amazon Cognito for authentication, created IAM execution roles, and learned how to generate and refresh bearer tokens for secure agent invocations.

### Next Steps

- **3: AgentCore Gateway** - Create gateway for tool exposure